# Notebook Leer Configuración

## Descripción General
Este notebook orquesta la ingesta de archivos Excel desde Azure Blob Storage hacia tablas Delta.
Soporta dos modos de ingesta:
- **Copia Completa**: Reemplazo completo de datos para conjuntos de datos snapshot
- **Incremental**: Operaciones de merge Delta para dimensiones de solo agregado o cambio lento

El notebook utiliza procesamiento paralelo mediante ThreadPool para ejecutar notebooks hijos concurrentemente.

## 1. Importaciones y Dependencias
Importar las bibliotecas Python requeridas para el manejo de fechas y procesamiento paralelo.

In [0]:
from datetime import datetime
from multiprocessing.pool import ThreadPool

## 2. Configuración de Parámetros de Widgets
Definir widgets de entrada para aceptar parámetros de tiempo de ejecución desde Azure Data Factory o ejecución manual.

| Parámetro | Descripción | Valor por Defecto |
|-----------|-------------|---------|
| Environment | Ambiente destino (dev, test, prod) | Vacío |
| IngestionGroup | Agrupación lógica de conjuntos de datos a procesar | Vacío |
| Datetime | Marca de tiempo de ejecución (opcional, por defecto UTC actual) | Vacío |

In [0]:
# Crear widgets de entrada para paso de parámetros externos (ej. desde Azure Data Factory)
dbutils.widgets.text("Environment", "", "Environment")
dbutils.widgets.text("IngestionGroup", "", "Ingestion Group")
dbutils.widgets.text("Datetime", "", "Datetime")

## 3. Configuración del Ambiente e Inicialización de Variables
Recuperar valores de widgets y configurar variables de configuración para el proceso de ingesta.

In [0]:
# Recuperar valores de parámetros desde los widgets
environment = dbutils.widgets.get("Environment")
ingestion_group = dbutils.widgets.get("IngestionGroup")

# Establecer fecha/hora de ejecución - usar valor proporcionado o por defecto la marca UTC actual
# Formato: YYYY-MM-DD HH:MM:SS
current_datetime = str(dbutils.widgets.get("Datetime") or datetime.now("UTC").strftime("%Y-%m-%d %H:%M:%S"))

# Definir rutas de notebooks hijos para diferentes estrategias de ingesta
fullcopy_notebook = 'Ingesta Completa'       # Notebook para operaciones de actualización completa
incremental_notebook = 'Ingesta Incremental' # Notebook para operaciones de merge delta

# Imprimir configuración para depuración y auditoría
print(environment, ingestion_group)

## 4. Carga de Configuración del Pipeline de Datos
Conectar a Azure SQL Database vía JDBC para obtener metadatos de ingesta desde la tabla ExcelConfiguration.

**Criterios de Filtrado:**
- Registros no eliminados lógicamente (`DeletedAt IS NULL`)
- Coincidentes con el Grupo de Ingesta especificado

**Estrategia de División:**
- Copia Completa: `IsIncremental = false` (cargas snapshot)
- Incremental: `IsIncremental = true` (operaciones CDC/merge)

In [0]:
# Recuperar cadena de conexión JDBC desde Azure Key Vault
# Alcance: kv-velman-dev, Clave: velkpi-sqlconnectionstr-jdbc
url = dbutils.secrets.get(scope="kv-velman-dev", key='velkpi-sqlconnectionstr-jdbc')

# Cargar datos de configuración desde tabla SQL con filtros
df_DataPipeline = (
    spark.read.jdbc(url, "velkpi.ExcelConfiguration")
    .filter("DeletedAt IS NULL")  # Excluir configuraciones eliminadas lógicamente
    .filter(f"IngestionGroup = '{ingestion_group}'")  # Filtrar por parámetro de grupo de ingesta
)

# Dividir dataframes por estrategia de ingesta
df_FullCopy = df_DataPipeline.filter("IsIncremental = false")    # Conjuntos de datos de actualización completa
df_Incremental = df_DataPipeline.filter("IsIncremental = true")  # Conjuntos de datos delta/incrementales

## 5. Procesamiento de Ingesta de Copia Completa
Ejecutar ingesta de copia completa en paralelo para todos los conjuntos de datos snapshot.

**Flujo del Proceso:**
1. Contar registros que requieren copia completa
2. Inicializar ThreadPool con tamaño igual al conteo de registros (un hilo por conjunto de datos)
3. Recolectar datos del lado del driver para iteración
4. Mapear función lambda para ejecutar notebook hijo para cada conjunto de datos
5. Pasar parámetros de configuración (rutas, rangos, filas a omitir, etc.)
6. Terminar pool después de la finalización

In [0]:
# Mostrar configuración de copia completa para monitoreo/depuración
display(df_FullCopy)

In [0]:
# Obtener conteo de conjuntos de datos de copia completa para determinar tamaño del pool de hilos
df_fullcopy_count = df_FullCopy.count()
print("Conteo de copia completa: " + str(df_fullcopy_count))

# Solo procesar si hay conjuntos de datos configurados para copia completa
if df_fullcopy_count > 0:
    # Inicializar ThreadPool para ejecución paralela (un hilo por conjunto de datos)
    fullcopy_pool = ThreadPool(df_fullcopy_count)

    # Recolectar datos de configuración al driver para iteración
    # Nota: collect() trae datos a la memoria del driver - adecuado para metadatos de configuración
    data_datapipeline = df_FullCopy.collect()

    # Ejecutar notebook hijo para cada conjunto de datos de copia completa en paralelo
    fullcopy_pool.map(
        lambda i: (
            dbutils.notebook.run(
                fullcopy_notebook,  # Ruta del notebook destino
                3600,               # Tiempo de espera en segundos (1 hora)
                {
                    # Parámetros principales
                    "Environment": str(environment or ""),                    # Ambiente destino
                    "ExcelPath": (i.BlobStagingPath + "/" + i.FileName),      # Ruta completa del blob al archivo Excel
                    "TableName": str(i.TableName or ""),                      # Nombre de la tabla Delta destino
                    "SheetName": str(i.SheetName or ""),                      # Nombre de la hoja de Excel
                    "DataRange": str(i.DataRange or ""),                      # Rango de celdas (ej. "A1:Z1000")
                    
                    # Parámetros de esquema y estructura
                    "IndexColumns": str(i.IndexColumns or ""),                # Columnas a usar como índice
                    "CustomColumnNames": str(i.CustomColumnNames or ""),      # Mapeos de encabezados personalizados
                    "ColumnsToSkip": str(i.ColumnsToSkip or ""),              # Columnas a excluir de la importación
                    "RowsToSkip": str(i.RowsToSkip or ""),                    # Filas de encabezado a omitir
                    
                    # Orientación de datos y formato
                    "IsHorizontal": str(i.IsHorizontal or ""),                # Diseño horizontal vs vertical
                    "MergeNHeaders": str(i.MergeNHeaders or ""),              # Número de filas de encabezado a fusionar
                    
                    # Parámetros de metadatos
                    "ReportDate": str(i.ReportDate or ""),                    # Fecha de negocio para los datos
                    "ForecastName": str(i.ForecastName or ""),                # Identificador de escenario de pronóstico
                    "Datetime": current_datetime,                             # Marca de tiempo de ejecución para auditoría
                },
            )
        ),
        data_datapipeline,  # Iterable de filas de configuración
    )
    
    # Limpiar recursos del pool de hilos
    fullcopy_pool.terminate()

## 6. Procesamiento de Ingesta Incremental
Ejecutar ingesta incremental en paralelo con detección de cambios basada en marca de agua.

**Parámetros Adicionales para Cargas Incrementales:**
- `IdColumns`: Columnas de clave primaria para coincidencia de merge
- `WatermarkColumn`: Columna usada para seguimiento de cambios (marca de tiempo/versión)
- `WatermarkLookBack`: Período de retrospectiva histórica para capturar llegadas tardías
- `WatermarkVal`: Último valor de marca de agua procesado (almacenado en configuración)

In [0]:
# Mostrar configuración incremental para monitoreo/depuración
display(df_Incremental)

In [0]:
# Obtener conteo de conjuntos de datos incrementales para determinar tamaño del pool de hilos
df_incremental_count = df_Incremental.count()
print(f"Conteo incremental: {df_incremental_count}")

# Solo procesar si hay conjuntos de datos configurados para carga incremental
if df_incremental_count > 0:
    # Inicializar ThreadPool para ejecución paralela
    incremental_pool = ThreadPool(df_incremental_count)
    
    # Recolectar datos de configuración al driver
    data_datapipeline = df_Incremental.collect()
    
    # Ejecutar notebook hijo para cada conjunto de datos incremental en paralelo
    incremental_pool.map(
        lambda i: (
            dbutils.notebook.run(
                incremental_notebook,  # Notebook destino para operaciones de merge delta
                3600,                  # Tiempo de espera en segundos (1 hora)
                {
                    # Identificación principal y ambiente
                    "Id": str(i.Id or ""),                                    # ID del registro de configuración
                    "Environment": str(environment or ""),                    # Ambiente destino
                    
                    # Ubicación de datos fuente
                    "ExcelPath": (i.BlobStagingPath + "/" + i.FileName),      # Ruta completa del blob
                    
                    # Definición de esquema destino
                    "TableName": str(i.TableName or ""),                      # Tabla Delta destino
                    "SheetName": str(i.SheetName or ""),                      # Hoja de Excel
                    "DataRange": str(i.DataRange or ""),                      # Especificación de rango de celdas
                    
                    # Configuración de columnas
                    "IndexColumns": str(i.IndexColumns or ""),                # Columnas de clave de negocio
                    "CustomColumnNames": str(i.CustomColumnNames or ""),      # Sobrescrituras de encabezados
                    "ColumnsToSkip": str(i.ColumnsToSkip or ""),              # Columnas excluidas
                    "RowsToSkip": str(i.RowsToSkip or ""),                    # Filas a ignorar al inicio
                    
                    # Diseño y formato
                    "IsHorizontal": str(i.IsHorizontal or ""),                # Bandera de orientación de datos
                    "MergeNHeaders": str(i.MergeNHeaders or ""),              # Manejo de encabezados multi-fila
                    
                    # Metadatos de negocio
                    "ReportDate": str(i.ReportDate or ""),                    # Fecha de período de reporte
                    "ForecastName": str(i.ForecastName or ""),                # Escenario de pronóstico
                    "Datetime": current_datetime,                             # Marca de tiempo de ejecución
                    
                    # Parámetros específicos de incrementales para lógica CDC/SCD
                    "IdColumns": str(i.IdColumns or ""),                      # Columnas de clave de merge (PK)
                    "WatermarkColumn": str(i.WatermarkColumn or ""),          # Columna de seguimiento de cambios
                    "WatermarkLookBack": str(i.WatermarkLookBack or ""),      # Ventana de retrospectiva (días)
                    "WatermarkVal": str(i.WatermarkVal or ""),                # Último valor procesado
                },
            )
        ),
        data_datapipeline,
    )
    
    # Limpiar recursos del pool de hilos para prevenir fugas de memoria
    incremental_pool.terminate()